<a href="https://colab.research.google.com/github/vanashri-18/CSA6101-Digital-Forensics-and-Cybercrime-Investigation/blob/main/Brute_Force_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**

To analyze a simulated authentication log using Python, group failed login attempts by username and IP address, and identify accounts or source IPs exceeding a configurable failure threshold within a selected time interval.

**Algorithm**

Generate or load the simulated authentication log.

Convert the timestamp column into datetime format.

Select only failed login attempts.

Group failures by Username and IP Address.

Define a configurable time interval and
failure threshold.

Count failed attempts occurring within the selected interval.

Identify groups exceeding the threshold.

Report the earliest attempt, latest attempt, total failures, username, and IP address.

Display the detected repeated-login-failure cases.

In [1]:
# Repeated Login Failure Detection
# Brute Force Detection

import pandas as pd
from datetime import datetime, timedelta

# --------------------------------------------------
# 1. Simulated Authentication Log
# --------------------------------------------------

data = [
    ["2026-08-25 09:00:00", "admin",  "192.168.1.10", "Failed"],
    ["2026-08-25 09:01:00", "admin",  "192.168.1.10", "Failed"],
    ["2026-08-25 09:02:00", "admin",  "192.168.1.10", "Failed"],
    ["2026-08-25 09:03:00", "admin",  "192.168.1.10", "Failed"],
    ["2026-08-25 09:04:00", "admin",  "192.168.1.10", "Failed"],
    ["2026-08-25 09:05:00", "admin",  "192.168.1.10", "Success"],

    ["2026-08-25 09:10:00", "user1",  "192.168.1.20", "Failed"],
    ["2026-08-25 09:12:00", "user1",  "192.168.1.20", "Failed"],
    ["2026-08-25 09:14:00", "user1",  "192.168.1.20", "Failed"],

    ["2026-08-25 09:20:00", "admin",  "10.0.0.5", "Failed"],
    ["2026-08-25 09:21:00", "admin",  "10.0.0.5", "Failed"],
    ["2026-08-25 09:22:00", "admin",  "10.0.0.5", "Failed"],
    ["2026-08-25 09:23:00", "admin",  "10.0.0.5", "Failed"],
]

df = pd.DataFrame(data, columns=[
    "Timestamp", "Username", "IP_Address", "Status"
])

df["Timestamp"] = pd.to_datetime(df["Timestamp"])

# --------------------------------------------------
# 2. Configurable Detection Parameters
# --------------------------------------------------

TIME_INTERVAL = 5       # minutes
FAILURE_THRESHOLD = 4   # minimum failures

# --------------------------------------------------
# 3. Select Failed Attempts
# --------------------------------------------------

failed = df[df["Status"].str.lower() == "failed"].copy()

# --------------------------------------------------
# 4. Detect Repeated Failures
# --------------------------------------------------

alerts = []

for (username, ip), group in failed.groupby(["Username", "IP_Address"]):

    group = group.sort_values("Timestamp").reset_index(drop=True)

    for i in range(len(group)):

        start_time = group.loc[i, "Timestamp"]
        end_time = start_time + timedelta(minutes=TIME_INTERVAL)

        window = group[
            (group["Timestamp"] >= start_time) &
            (group["Timestamp"] <= end_time)
        ]

        if len(window) >= FAILURE_THRESHOLD:

            alerts.append({
                "Username": username,
                "IP_Address": ip,
                "Earliest_Attempt": window["Timestamp"].min(),
                "Latest_Attempt": window["Timestamp"].max(),
                "Total_Failures": len(window)
            })

            break

# --------------------------------------------------
# 5. Display Results
# --------------------------------------------------

result = pd.DataFrame(alerts)

print("=" * 70)
print("       REPEATED LOGIN FAILURE / BRUTE FORCE REPORT")
print("=" * 70)

print(f"Time Interval : {TIME_INTERVAL} minutes")
print(f"Failure Threshold : {FAILURE_THRESHOLD}")
print()

if len(result) > 0:
    print("Suspicious login activity detected!\n")
    print(result.to_string(index=False))
else:
    print("No repeated login failure detected.")

       REPEATED LOGIN FAILURE / BRUTE FORCE REPORT
Time Interval : 5 minutes
Failure Threshold : 4

Suspicious login activity detected!

Username   IP_Address    Earliest_Attempt      Latest_Attempt  Total_Failures
   admin     10.0.0.5 2026-08-25 09:20:00 2026-08-25 09:23:00               4
   admin 192.168.1.10 2026-08-25 09:00:00 2026-08-25 09:04:00               5


**Result:**
The raw email headers were successfully parsed using Python, and the From, Reply-To, Return-Path, Received, SPF, DKIM, and DMARC fields were extracted where available. The program compared these fields and classified the observations as Normal Observation, Review, or Suspicious Indicator. Sender mismatches were not automatically treated as malicious, while multiple authentication failures combined with inconsistencies were reported as suspicious indicators. This demonstrates a practical approach to identifying possible email spoofing.